In [19]:
import torch
import transformer_lens
from sae_lens import SAE


In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [21]:
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.6.hook_resid_pre",
    device="cuda"
)

In [22]:
model = transformer_lens.HookedTransformer.from_pretrained("gpt2-small",  device=device, use_cache=True)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11422.31it/s]


Loaded pretrained model gpt2-small into HookedTransformer


In [23]:
import json

with open('deepseek.json') as f:
    data = json.load(f)['development']
print(len(data))

20


In [24]:
alphas = [0.0, 5.0, 10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,19.0, 20.0]

ids = [4192, 20026, 4214, 8301, 19470, 4262, 13374, 15462, 12552]

In [25]:
lt_tns = torch.randn(1,1,768).to(device)
se_encd = sae.encode(lt_tns)
se_encd.shape

torch.Size([1, 1, 24576])

In [26]:

def hook_steering(tensor, hook):
    last_tensor =  tensor[:,-1,:].to(device)
    sae_encode = sae.encode(last_tensor)
    sae_encode[:,idx] += alpha
    sae_decod = sae.decode(sae_encode).unsqueeze(0).to(device)
    result = torch.concat([tensor[:,:-1,:], sae_decod], dim=1)
    return result

In [27]:
idx = ids[2]

for idx_p, prompt in enumerate(data):
    for alpha in alphas:
        res = []
        print(f'prompt: {idx_p}')
        for i in range(10):
            torch.manual_seed(42 + i + idx_p)
            no_steering = model.generate(prompt,temperature=0.9, max_new_tokens=100)

            model.add_hook(name="blocks.6.hook_resid_pre", hook = hook_steering, dir='fwd')
            torch.manual_seed(42 + i + idx_p)
            steering = model.generate(prompt,temperature=0.9, max_new_tokens=100)
            model.reset_hooks()
            res.append({
                'prompt': prompt,
                'no_steering': no_steering,
                'steering': steering,
                'alpha': alpha,
                'seed': 42 + i + idx_p
            })
        with open(f'res_sae/1/{idx_p}_{alpha}.json', mode='w') as f:
            json.dump(res, f)

prompt: 0


100%|██████████| 100/100 [00:01<00:00, 56.78it/s]


prompt: 0


100%|██████████| 100/100 [00:01<00:00, 57.30it/s]


prompt: 0


...

100%|██████████| 100/100 [00:01<00:00, 88.85it/s]


prompt: 19


100%|██████████| 100/100 [00:01<00:00, 87.41it/s]


prompt: 19


 42%|████▏     | 42/100 [00:00<00:00, 87.97it/s]
